# Taller 1 · EcoMarket — integración grupal de ambas propuestas
# Ver demo 9 y 5
Eres el equipo de IA de **EcoMarket**, un supermercado online. Tu misión: construir en **LangGraph** el agente que atiende a los clientes por chat. El código base ya trae la estructura completa — tú completas los nodos que le dan vida.

##  Lo que tu agente debe saber hacer

| # | Flujo | Qué hace | Estado |
|---|---|---|---|
| 1 | **Recetas** | Recomienda una receta (con link) y arma la lista de compra con precios de la tienda | **TODO 1** |
| 2 | **Estado de pedidos** | Informa cómo va un pedido por su número | **`TODO` 2** |
| 3 | **Horarios** | Responde horarios de despacho por comuna | **TODO 3** |
| 4 | **Conversacional** | Conversa con memoria del chat | **TODO 4** |
| 5 | **Tu flujo ++** | Uno inventado por ti (ofertas, reclamos, maridaje…) | **BONUS** |

Un **router** (como el de la Demo 9) clasifica cada mensaje y delega al flujo correcto. Las **tools** (recetas con URLs simuladas, catálogo, pedidos, horarios) ya están listas — solo tienes que **usarlas**.

##  Cómo trabajar (paso a paso)

1. **Copia el código a tu espacio**: clic en el editor → `Ctrl/Cmd+A` → copiar → abre **Mi laboratorio** (arriba en el sidebar) → pegar.
2. **Ejecuta tal cual**: el agente YA corre — saluda por el flujo conversacional y los TODO responden «pendiente». Mira el diagrama: el router ya enruta.
3. **Completa TODO 1, 2 y 3**: cada uno trae una **PISTA** con las líneas clave. El nodo `conversar` (ya resuelto) es tu ejemplo del patrón: *contexto → llm.invoke → {"respuesta": texto(r)}*.
4. **Prueba cada flujo** con la pestaña **Preguntas** (botón «Al chat») y verifica en el **diagrama** que el router tomó la ruta correcta — clic en cada nodo para ver su entrada/salida.
5. **Afina sin código**: en la pestaña **Prompts** puedes cambiar la personalidad del chef, el criterio del router, etc.
6. **BONUS (TODO 5)**: agrega un flujo nuevo — las 4 cosas que hay que tocar están comentadas al final del código.

##  Entrega · lee esto antes de descargar

Se entrega el **Notebook .ipynb** (botón arriba a la derecha). Para que sea evaluado tiene que cumplir **las tres**:

1. **Nombres de los integrantes** escritos en la lista `INTEGRANTES` de la celda **ENTREGA** (la primera celda de código después de la clave).
2. **Tus prompts dentro del notebook**: pega el texto final de `PROMPT_ROUTER`, `PROMPT_CHEF` y `PROMPT_ASISTENTE` en la celda **ENTREGA**. La pestaña Prompts de la plataforma **no viaja** con el notebook: si no los pegas, el agente no rutea ni cocina y la nota es 0 en esos flujos.
3. **Todo el código corre** de arriba abajo sin errores: antes de enviar, `Kernel → Restart & Run All` (en Colab: *Entorno de ejecución → Reiniciar y ejecutar todo*). Un notebook con una celda rota no se evalúa.

La celda ENTREGA imprime `✅ Entrega: …` cuando las tres cosas están, y `⚠️ ENTREGA INCOMPLETA` si falta algo.

### Qué se envía: un solo `.zip`

Se entrega **un archivo `apellido_nombre.zip`** por el canal del curso con esto adentro:

```
apellido_nombre.zip
├── taller1.ipynb          ← el notebook con nombres, prompts y todo corriendo
├── plan.json              ← el plan de tareas que generó @spec y que @implementador fue marcando como hechas
└── .opencode/
    └── agents/
        ├── spec.md         ← tus agentes de OpenCode tal como los usaste
        └── implementador.md
```

El `plan.json` y la carpeta `.opencode/` son parte de la evaluación: muestran **cómo** llegaste al resultado (qué tareas pediste, cuáles quedaron hechas y con qué verificación). Un zip sin `plan.json` o sin `.opencode/` se considera incompleto.

Para armarlo: en la carpeta del proyecto, Mac/Linux `zip -r apellido_nombre.zip taller1.ipynb plan.json .opencode`; Windows: selecciona los tres, clic derecho → *Comprimir en archivo ZIP* (la carpeta `.opencode` está oculta: activa *Ver → Elementos ocultos*). No incluyas `.venv` ni `.env`.

##  Rúbrica

| Criterio | Puntos |
|---|---|
| Flujo de recetas funciona (usa las 2 tools, receta + lista con precios) | 25 |
| Flujo de pedidos funciona (extrae el número, maneja pedido inexistente) | 20 |
| Flujo de horarios funciona | 15 |
| El router enruta bien las preguntas de la pestaña Preguntas | 15 |
| Conversación con memoria intacta (no romper lo que ya funciona ) | 10 |
| **Bonus**: flujo adicional completo (tool + nodo + ruta) | +15 |

##  Tips

- Las tools se llaman con `.invoke({...})`: `buscar_recetas.invoke({"ingrediente": "palta"})`.
- Si el router se equivoca de ruta, el problema casi siempre es `PROMPT_ROUTER` — arréglalo desde la pestaña Prompts.
- Prueba también los bordes: un pedido que no existe (`#9999`), una comuna sin despacho.
- El pedido del cliente completo viene en `estado["entrada"]` — pásaselo al LLM para que la respuesta sea pertinente.

---
**Integración grupal:** versión combinada a partir de `Munoz_Matias.zip` y `basaure_eduardo.zip`. Conserva el catálogo simulado original; las ofertas son una extensión ficticia, no promociones reales. Ver `plan.json` para procedencia de evidencias y pruebas pendientes.


In [ ]:
# Dependencias de esta clase. Tarda ~1 min la primera vez.
# Si igual aparece un aviso rojo de google-colab / google-auth, ignoralo: es normal y no afecta.
import sys
import subprocess
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "langchain-core", "langchain-google-genai", "langgraph", "pydantic"],
    shell=False,
    check=True,
)
print("Dependencias listas")


In [ ]:
# Tu API key de Gemini (gratis en https://aistudio.google.com/apikey).
# Al ejecutar esta celda aparece un cuadro: pega la key ahi (no se muestra ni queda guardada en el notebook).
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")
print("API key lista")


### Ayudantes de la plataforma
Esta celda crea `plataforma.py` (chat, diagrama, timeline, trazar) para que el código de la clase funcione fuera de la web. En el notebook, `chat()` y los diagramas se imprimen como texto.


In [ ]:
%%writefile plataforma.py
"""Ayudantes de la plataforma — importables desde el código de las clases.

    from plataforma import chat, diagrama, trazar

- chat(texto)                → manda texto al chat de la web
- diagrama(nodos, aristas)   → dibuja un diagrama hecho a mano
- trazar(app, estado, ...)   → ejecuta un grafo de LangGraph y dibuja
                               EL PASO A PASO REAL en la pestaña Diagrama
- timeline(pasos)            → dibuja pasos de un agente escrito a mano
                               (sin LangGraph): [("nodo", "resumen"), ...]
- texto(msg)                 → aplana el content de un mensaje del LLM
                               (str, lista de partes o vacío) a texto plano
- imagen(png)                → muestra un PNG en el chat (bytes o base64);
                               print(app.get_graph().draw_mermaid_png())
                               también funciona directo
- memoria_guardar(nombre, datos) / memoria_cargar(nombre, default)
                             → persistencia JSON por sección (sobrevive
                               reinicios): memoria episódica, índices, etc.
- archivos()                 → rutas de los archivos subidos a la sección
- leer_pdf(ruta)             → páginas de texto de un PDF [{pagina, texto}]
"""
import base64 as _base64
import builtins as _builtins
import json as _json

_print_original = print


def chat(texto):
    """Manda texto al chat de la web (una línea por ##CHAT##)."""
    for linea in str(texto).splitlines():
        if linea.strip():
            _print_original("##CHAT## " + linea)


def diagrama(nodos, aristas):
    """Dibuja un diagrama hecho a mano: listas de nodos y aristas React Flow."""
    _print_original("##STATE##" + _json.dumps({"nodes": nodos, "edges": aristas}, ensure_ascii=False))


def imagen(datos, alt="diagrama"):
    """Muestra una imagen PNG en el chat de la web.

    Acepta los bytes de un PNG (p. ej. app.get_graph().draw_mermaid_png())
    o un string ya codificado en base64."""
    if isinstance(datos, (bytes, bytearray)):
        datos = _base64.b64encode(bytes(datos)).decode()
    _print_original("##IMG##" + str(datos).strip())


# ── 🧠 Memoria persistente y archivos de la sección ─────────────────────────
import os as _os
import pathlib as _pathlib

def _memoria_dir():
    d = _pathlib.Path(_os.environ.get("MEMORIA_DIR", "./memoria"))
    d.mkdir(parents=True, exist_ok=True)
    return d


def memoria_guardar(nombre, datos):
    """Guarda `datos` (cualquier JSON) con ese nombre. Sobrevive reinicios y
    es compartido por toda la sección: úsalo para memoria episódica,
    índices vectoriales, grafos de entidades, contadores..."""
    ruta = _memoria_dir() / (str(nombre).replace("/", "_") + ".json")
    ruta.write_text(_json.dumps(datos, ensure_ascii=False, indent=1), encoding="utf-8")
    return str(ruta)


def memoria_cargar(nombre, default=None):
    """Recupera lo guardado con memoria_guardar (o `default` si no existe)."""
    ruta = _memoria_dir() / (str(nombre).replace("/", "_") + ".json")
    if not ruta.exists():
        return default
    try:
        return _json.loads(ruta.read_text(encoding="utf-8"))
    except Exception:
        return default


def memoria_borrar(nombre):
    """Elimina una memoria guardada (para empezar de cero)."""
    ruta = _memoria_dir() / (str(nombre).replace("/", "_") + ".json")
    if ruta.exists():
        ruta.unlink()
        return True
    return False


def archivos(extension=None):
    """Rutas de los archivos subidos a la sección (botón Archivos de la web).
    extension: filtra por ".pdf", ".txt", ..."""
    d = _pathlib.Path(_os.environ.get("ARCHIVOS_DIR", "./memoria/archivos"))
    if not d.exists():
        return []
    rutas = sorted(p for p in d.iterdir() if p.is_file())
    if extension:
        rutas = [p for p in rutas if p.suffix.lower() == extension.lower()]
    return [str(p) for p in rutas]


def leer_pdf(ruta):
    """Extrae el texto de un PDF por páginas: [{"pagina": 1, "texto": "..."}, ...].
    Para .txt/.md devuelve una sola "página" con todo el texto."""
    ruta = _pathlib.Path(ruta)
    if ruta.suffix.lower() != ".pdf":
        return [{"pagina": 1, "texto": ruta.read_text(encoding="utf-8", errors="ignore")}]
    from pypdf import PdfReader
    paginas = []
    for i, p in enumerate(PdfReader(str(ruta)).pages, 1):
        t = (p.extract_text() or "").strip()
        if t:
            paginas.append({"pagina": i, "texto": t})
    return paginas


def _es_png(x):
    """True si x son los bytes de un PNG (firma 89 P N G)."""
    return isinstance(x, (bytes, bytearray)) and bytes(x[:4]) == bytes([137, 80, 78, 71])


def _print_plataforma(*args, **kwargs):
    """print() mejorado: si imprimes los bytes de un PNG — por ejemplo
    print(app.get_graph().draw_mermaid_png()) — la imagen aparece en el chat
    en vez de llenar la terminal de bytes."""
    if kwargs.get("file") is not None:
        return _print_original(*args, **kwargs)
    resto = [a for a in args if not _es_png(a)]
    for a in args:
        if _es_png(a):
            imagen(a)
    if resto or not args:
        _print_original(*resto, **kwargs)


_builtins.print = _print_plataforma


def _texto_de(x):
    """Extrae texto plano de lo que sea: str, mensaje de LangChain o lista de partes."""
    c = getattr(x, "content", x)
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)
    return "" if c is None else str(c)


def texto(msg) -> str:
    """Aplana el content de una respuesta del LLM a texto plano.

    Gemini (y otros) pueden devolver content como str, lista de partes
    [{"type": "text", "text": ...}, ...] o vacío — esto lo normaliza.
    Uso:  respuesta = texto(llm.invoke(...))
    """
    return _texto_de(msg)


def _plano(x, tope=2000):
    """Convierte cualquier estado/update en algo serializable y legible
    (los mensajes de LangChain se muestran como [rol] texto)."""
    if isinstance(x, dict):
        return {str(k): _plano(v, tope) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_plano(v, tope) for v in x]
    if isinstance(x, str):
        return x if len(x) <= tope else x[:tope] + " …(recortado)"
    if isinstance(x, (int, float, bool)) or x is None:
        return x
    if hasattr(x, "content"):
        rol = getattr(x, "type", x.__class__.__name__)
        return _plano(f"[{rol}] {_texto_de(x)}", tope)
    return _plano(str(x), tope)


def _resumen(update, limite=30, previo=None):
    """Un vistazo corto de lo que devolvió un nodo (para la etiqueta de la arista).

    Prefiere lo que el nodo produjo de NUEVO respecto al estado previo:
    el elemento recién agregado a una lista o el texto que cambió — así un
    plan que se consume no repite la misma etiqueta en cada paso."""
    if not isinstance(update, dict):
        return ""
    previo = previo if isinstance(previo, dict) else {}
    candidatos = []  # (prioridad, orden, texto): 0=contenido nuevo ... 3=sin novedad
    for orden, (k, v) in enumerate(update.items()):
        antes = previo.get(k)
        if isinstance(v, list):
            if not v:
                continue
            lista_antes = antes if isinstance(antes, list) else []
            if len(v) > len(lista_antes) and v[:len(lista_antes)] == lista_antes:
                s, prio = _texto_de(v[len(lista_antes)]), 0  # la lista creció: lo nuevo
            else:
                # con reducers (operator.add) el update trae SOLO el delta:
                # cualquier elemento que no estaba antes es contenido nuevo
                nuevos = [x for x in v if x not in lista_antes]
                if nuevos:
                    s, prio = _texto_de(nuevos[-1]), 0
                else:
                    s, prio = _texto_de(v[-1]), 3  # plan consumido / sin novedad
        elif isinstance(v, (str, int, float)) or hasattr(v, "content"):
            s, prio = _texto_de(v), (1 if antes != v else 2)
        else:
            continue
        s = " ".join(str(s).split())
        if s:
            candidatos.append((prio, orden, s))
    if not candidatos:
        return ""
    s = min(candidatos)[2]
    return s[:limite] + "…" if len(s) > limite else s


def timeline(pasos, titulo_final="💬 Fin", estado_inicial=None, estado_final=None):
    """Dibuja una secuencia de pasos ejecutados como diagrama (sin LangGraph).

    pasos: lista de nombres, tuplas (nombre, resumen), (nombre, resumen, icono)
    o (nombre, resumen, icono, detalle) con detalle={"entrada": ..., "salida": ...}
    — en la web, clic en el nodo abre un modal con esa entrada/salida.
    Ideal para agentes escritos a mano: ve agregando ("nodo", "qué pasó")
    a una lista y llama timeline(pasos) al final.
    """
    dato_in = {"detalle": {"salida": _plano(estado_inicial)}} if estado_inicial is not None else {}
    nodes = [{"id": "in", "data": {"label": "👤 Entrada", "role": "input", **dato_in},
              "position": {"x": 0, "y": 40}}]
    edges = []
    prev = "in"
    for i, paso in enumerate(pasos, 1):
        item = list(paso) if isinstance(paso, (list, tuple)) else [paso]
        item += [""] * (4 - len(item))
        nombre, resumen, icono, detalle = item[0], item[1], item[2] or "⚙️", item[3] or None
        resumen = " ".join(str(resumen).split())[:24]
        nid = f"s{i}"
        col, fila = (i - 1) % 3, (i - 1) // 3
        nodes.append({"id": nid, "data": {"label": f"{icono} {i}. {nombre}",
                                          "role": "tool" if icono == "🔨" else "process",
                                          **({"detalle": _plano(detalle)} if detalle else {})},
                      "position": {"x": 220 + col * 230, "y": 40 + fila * 110}})
        edges.append({"id": f"e{i}", "source": prev, "target": nid, "animated": True,
                      **({"label": resumen} if resumen else {})})
        prev = nid
    dato_out = {"detalle": {"entrada": _plano(estado_final)}} if estado_final is not None else {}
    nodes.append({"id": "out", "data": {"label": titulo_final, "role": "output", **dato_out},
                  "position": {"x": 220 + 3 * 230, "y": 40}})
    edges.append({"id": "efin", "source": prev, "target": "out", "animated": True})
    diagrama(nodes, edges)


def _diagrama_grupos(grupos, estado_inicial=None, estado_final=None):
    """Dibuja un flujo con ramas PARALELAS: cada grupo es una columna y los
    pasos de la misma columna corrieron a la vez (mismo superstep). Las
    aristas hacen el fan-out / fan-in real entre columnas."""
    dato_in = {"detalle": {"salida": _plano(estado_inicial)}} if estado_inicial is not None else {}
    nodes = [{"id": "in", "data": {"label": "👤 Entrada", "role": "input", **dato_in},
              "position": {"x": 0, "y": 40}}]
    edges = []
    previos = ["in"]
    n = 0
    for col, grupo in enumerate(grupos):
        ids = []
        for fila, paso in enumerate(grupo):
            item = list(paso) if isinstance(paso, (list, tuple)) else [paso]
            item += [""] * (4 - len(item))
            nombre, resumen, icono, detalle = item[0], item[1], item[2] or "⚙️", item[3] or None
            resumen = " ".join(str(resumen).split())[:24]
            n += 1
            nid = f"s{n}"
            y = 40 + int((fila - (len(grupo) - 1) / 2) * 110)
            nodes.append({"id": nid, "data": {"label": f"{icono} {n}. {nombre}",
                                              "role": "tool" if icono == "🔨" else "process",
                                              **({"detalle": _plano(detalle)} if detalle else {})},
                          "position": {"x": 220 + col * 240, "y": y}})
            for i, p in enumerate(previos):
                edges.append({"id": f"e{p}-{nid}", "source": p, "target": nid, "animated": True,
                              **({"label": resumen} if resumen and i == 0 else {})})
            ids.append(nid)
        previos = ids
    dato_out = {"detalle": {"entrada": _plano(estado_final)}} if estado_final is not None else {}
    nodes.append({"id": "out", "data": {"label": "💬 Fin", "role": "output", **dato_out},
                  "position": {"x": 220 + len(grupos) * 240, "y": 40}})
    for p in previos:
        edges.append({"id": f"e{p}-out", "source": p, "target": "out", "animated": True})
    diagrama(nodes, edges)


def _uso_tool(update):
    """True si el update de este nodo contiene mensajes de herramienta."""
    if isinstance(update, dict):
        for v in update.values():
            if isinstance(v, list):
                for x in v:
                    if getattr(x, "type", "") == "tool":
                        return True
    return False


def trazar(app, estado_inicial, responder_desde=None, config=None, detalle=True):
    """Ejecuta un grafo de LangGraph dibujando el paso a paso real.

    Cada nodo que se ejecuta aparece como un paso numerado en el diagrama
    (los loops se ven como visitas repetidas); los pasos que ejecutan
    herramientas se marcan con 🔨. Devuelve el estado final.
    Si responder_desde="campo", ese campo del estado final va al chat.
    En la web, clic en un nodo del diagrama abre un modal con la ENTRADA
    (estado que recibió) y la SALIDA (lo que devolvió) de ese paso.
    """
    grupos = []   # cada grupo = pasos que corrieron EN PARALELO (mismo superstep)
    actual = []
    total = 0
    final = estado_inicial
    previo = estado_inicial  # estado ANTES del paso en curso (para el modal)
    for modo, chunk in app.stream(estado_inicial, config, stream_mode=["updates", "values"]):
        if modo == "values":
            final = chunk
            previo = chunk
            if actual:
                grupos.append(actual)
                actual = []
        elif modo == "updates":
            for nodo, update in (chunk or {}).items():
                if str(nodo).startswith("__"):
                    continue
                icono = "🔨" if _uso_tool(update) else ""
                es = {"entrada": _plano(previo), "salida": _plano(update)}
                actual.append((str(nodo), _resumen(update, previo=previo) if detalle else "", icono, es))
                total += 1
                print(f"[trazar] paso {total}: {nodo}" + (" (tool)" if icono else ""))
    if actual:
        grupos.append(actual)

    if all(len(g) == 1 for g in grupos):
        timeline([g[0] for g in grupos], estado_inicial=estado_inicial, estado_final=final)
    else:
        _diagrama_grupos(grupos, estado_inicial=estado_inicial, estado_final=final)

    if responder_desde and isinstance(final, dict) and final.get(responder_desde):
        chat(final[responder_desde])
    return final


In [ ]:
"""TALLER · Generador de Recetas para el ecommerce "EcoMarket".

Tu agente atiende clientes del supermercado por chat. Debe saber:
  1. Recomendar RECETAS con productos de la tienda (tools simuladas)
  2. Informar el ESTADO de un PEDIDO
  3. Responder HORARIOS de despacho
  4. CONVERSAR 
  5. BONUS: un flujo nuevo inventado por ti

COMO TRABAJAR (detalle completo en la pestana Docs):
  A. Descarga El notebook.
  B. Corre Celda por celda del notebook.
  C. Completa TODO 1, 2 y 3 imitando el nodo `conversar`. Cada uno trae PISTA.
  D. Prueba cada flujo con la pestana Preguntas y mira el diagrama.
  E. Entrega: un .zip con el notebook (nombres de integrantes, prompts pegados en la celda
     ENTREGA y TODO el codigo corriendo), mas plan.json y la carpeta .opencode/.
"""
import os
import re
from typing import TypedDict, Literal

from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

from plataforma import chat, texto, trazar

# ── La key viene del panel API keys (o backend/.env) — NUNCA del código ────
if not os.environ.get("GOOGLE_API_KEY"):
    print(" Falta **GOOGLE_API_KEY**. Agrégala en el panel *API keys* y vuelve a ejecutar.")
    raise SystemExit(0)

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_retries=0)


In [ ]:
# ═══════════════════════ ENTREGA · OBLIGATORIO PARA LA EVALUACIÓN ═══════════════════════
# 1. Escribe los nombres REALES de los integrantes del equipo.
# 2. Pega aquí tus prompts finales. En el notebook NO existen las variables de la pestaña Prompts
#    de la plataforma: si los dejas vacíos, el agente no rutea ni cocina y la evaluación falla.
# 3. Antes de enviar: Kernel → Restart & Run All. TODO el notebook debe correr sin errores.
# 4. Se entrega UN .zip (apellido_nombre.zip) con: este notebook, plan.json y la carpeta .opencode/ (spec.md e implementador.md).
INTEGRANTES = ["Eduardo Basaure", "Matías Muñoz", "Alex Nail", "Sebastián Mardones"]

PROMPT_ROUTER_ENTREGA = """Eres el clasificador de intención de EcoMarket. No respondas al cliente ni ejecutes acciones: selecciona exactamente un destino permitido por el esquema Ruta: receta, pedido, horarios, conversar u ofertas. Usa el sentido completo del mensaje, no una palabra aislada.

Criterios:
- receta: solicita recomendar, elegir o preparar una receta/comida, o una lista de compra para cocinar; incluye ingredientes, preferencias y restricciones culinarias. No uses esta ruta para una pregunta de precio/disponibilidad sin una petición culinaria.
- pedido: solicita explícitamente consultar, rastrear o explicar el estado de un pedido/orden/compra ya realizada (por ejemplo, “¿dónde está mi pedido #1042?”). Un número aislado puede ser el identificador. Un número que claramente expresa cantidad, precio, fecha, dirección u otro dato no es un ID de pedido.
- horarios: pregunta por días, horas, cobertura o disponibilidad de despacho para una comuna, también si nombra más de una comuna o una comuna posiblemente no cubierta. Una pregunta por el estado de un pedido concreto corresponde a pedido, no a horarios.
- ofertas: solicita explicitamente consultar ofertas, promociones o descuentos. No uses esta ruta para una consulta de precio regular sin peticion de promocion. Si tambien pide una receta u otra accion independiente y no queda clara la prioridad, usa conversar para pedir aclaracion.
- conversar: saludos, despedidas, preguntas generales sobre el asistente o EcoMarket, preguntas comerciales que no tienen flujo de atención específico, mensajes sin intención clara y solicitudes ambiguas.

Desempate:
1. Clasifica según lo que el cliente pide que se haga, no solo por términos mencionados.
2. Si hay dos solicitudes independientes de rutas distintas o no se puede determinar la intención principal, elige conversar para pedir aclaración; no priorices arbitrariamente.
3. Si el número aparece como cantidad (“para 2”), año, precio u otro número no asociado a un pedido, no elijas pedido por esa cifra.
4. Si el mensaje es solo un identificador numérico o #numérico, elige pedido.
5. Devuelve únicamente la clasificación estructurada Ruta.destino, sin explicación ni texto adicional."""
PROMPT_CHEF_ENTREGA = """Eres el selector de recetas de EcoMarket. Considera la solicitud completa del cliente y elige únicamente una receta cuyo nombre aparezca exactamente en las recetas devueltas por la herramienta. Basa la elección solo en los ingredientes, el tiempo y demás datos explícitamente proporcionados por esa herramienta. No infieras ni afirmes atributos, ingredientes, cantidades, disponibilidad u otras características que esos datos no respalden. El catálogo de precios es información de referencia; no inventes ni calcules precios. Si ninguna opción satisface plenamente las restricciones expresas, responde SIN_COINCIDENCIA para solicitar aclaración, sin presentar coincidencias no verificadas como válidas. Responde únicamente con el nombre exacto de la receta elegida, sin URL, explicación, lista de compra ni información adicional."""
PROMPT_ASISTENTE_ENTREGA = """Eres el asistente conversacional de atención al cliente de EcoMarket, un supermercado. Responde en español, con cortesía, claridad y brevedad. Atiende saludos, preguntas generales y conversación; usa CHAT_HISTORY como contexto de continuidad sin repetirlo ni tratarlo como fuente comercial verificada.

No inventes ni supongas productos, precios, existencias, descuentos, pedidos, estados, fechas, transportistas, comunas, horarios, políticas ni capacidades. En esta ruta no tienes resultados de herramientas de catálogo, pedidos o despacho: si preguntan por esos datos, indica con transparencia que no puedes confirmarlos aquí y solicita una aclaración pertinente o explica brevemente qué información falta. No finjas haber consultado sistemas ni prometas acciones. Si la petición es ambigua, haz una pregunta concreta antes de asumir. Si preguntan por asuntos externos a EcoMarket (noticias, clima, UF o temas ajenos), explica brevemente el límite y ofrece ayuda con los servicios del supermercado. No des asesoría profesional como si fuera información oficial de EcoMarket."""

# En la plataforma mandan los de la pestaña Prompts; en el notebook, los que pegaste arriba.
PROMPT_ROUTER = os.environ.get("PROMPT_ROUTER") or PROMPT_ROUTER_ENTREGA        # Router
PROMPT_CHEF = os.environ.get("PROMPT_CHEF") or PROMPT_CHEF_ENTREGA              # Recetas
PROMPT_ASISTENTE = os.environ.get("PROMPT_ASISTENTE") or PROMPT_ASISTENTE_ENTREGA  # Conversacion

_faltan = [n for n, v in (("INTEGRANTES", INTEGRANTES == ["Nombre Apellido 1", "Nombre Apellido 2"]),
                          ("PROMPT_ROUTER", "PEGA AQUÍ" in PROMPT_ROUTER), ("PROMPT_CHEF", "PEGA AQUÍ" in PROMPT_CHEF),
                          ("PROMPT_ASISTENTE", "PEGA AQUÍ" in PROMPT_ASISTENTE)) if v]
if _faltan:
    print("⚠️  ENTREGA INCOMPLETA: falta completar " + ", ".join(_faltan) + ". Sin esto el taller NO se evalúa.")
else:
    print("✅ Entrega: " + ", ".join(INTEGRANTES) + " · prompts cargados")

## 
os.environ.setdefault("CHAT_HISTORY", "[]")


In [ ]:
# ═══════════════ DATOS SIMULADOS DE ECOMARKET (no los edites) ══════════════
RECETAS = [
    {"nombre": "Pasta al pesto con palta", "url": "https://recetas.ecomarket.cl/pasta-pesto-palta",
     "ingredientes": ["pasta", "palta", "albahaca", "queso"], "minutos": 20},
    {"nombre": "Tacos veganos de quinoa", "url": "https://recetas.ecomarket.cl/tacos-quinoa",
     "ingredientes": ["quinoa", "tortillas", "palta", "tomate"], "minutos": 25},
    {"nombre": "Pan con palta de campeonato", "url": "https://recetas.ecomarket.cl/pan-palta",
     "ingredientes": ["pan", "palta", "limon", "sal de mar"], "minutos": 5},
    {"nombre": "Salmon al horno con quinoa", "url": "https://recetas.ecomarket.cl/salmon-quinoa",
     "ingredientes": ["salmon", "quinoa", "limon", "eneldo"], "minutos": 35},
    {"nombre": "Ensalada de tomate y albahaca", "url": "https://recetas.ecomarket.cl/ensalada-caprese",
     "ingredientes": ["tomate", "albahaca", "queso", "aceite de oliva"], "minutos": 10},
]
CATALOGO = {"pasta": 1490, "palta": 3990, "albahaca": 990, "queso": 4590, "quinoa": 3290,
            "tortillas": 2190, "tomate": 1890, "pan": 1990, "limon": 1290, "sal de mar": 890,
            "salmon": 8990, "eneldo": 790, "aceite de oliva": 5990}
# Fictitious test fixture only; not a real/current commercial promotion.
# Integración del bonus de los compañeros (3 casos). Datos solo para simulación,
# no ofertas reales ni certificaciones dietarias.
OFERTAS_SIMULADAS = [
    {"producto": "palta", "descuento_pct": 15, "etiquetas_prueba": []},
    {"producto": "quinoa", "descuento_pct": 10, "etiquetas_prueba": ["vegetariana"]},
    {"producto": "tomate", "descuento_pct": 20, "etiquetas_prueba": []},
]
PEDIDOS = {
    "1042": {"estado": "en reparto", "detalle": "sale hoy entre 16:00 y 18:00", "items": 8},
    "1043": {"estado": "preparando", "detalle": "en bodega, despacho manana", "items": 3},
    "2001": {"estado": "entregado", "detalle": "recibido ayer 19:24 por J. Perez", "items": 12},
}
HORARIOS = {
    "providencia": "lun-sab 10:00-20:00, mismo dia si compras antes de las 14:00",
    "nunoa": "lun-sab 10:00-20:00, mismo dia si compras antes de las 14:00",
    "maipu": "lun-vie 12:00-21:00, despacho al dia siguiente",
    "las condes": "todos los dias 9:00-22:00, express en 90 minutos",
}


In [ ]:
# ═══════════════ TOOLS (ya listas: solo tienes que USARLAS) ════════════════
def _normalizar_recetas(valor: str) -> str:
    """Normaliza texto sin alterar los datos originales del catálogo."""
    import unicodedata
    descompuesto = unicodedata.normalize("NFD", str(valor).casefold())
    return " ".join("".join(c for c in descompuesto if unicodedata.category(c) != "Mn").split())


def _formas_ingrediente_receta(nombre: str) -> set[str]:
    """Variaciones comunes, sin inferir ingredientes que no existen."""
    base = _normalizar_recetas(nombre)
    formas = {base, base + "s", base + "es"}
    if base.endswith("s"):
        formas.add(base[:-1])
    if " " in base:
        primero, resto = base.split(" ", 1)
        formas.add(primero + "s " + resto)
        formas.add(primero + "es " + resto)
    return formas


def _ingredientes_receta_en_consulta(consulta: str) -> tuple[set[str], set[str]]:
    """Reconoce nombres del catálogo y exclusiones expresas como 'sin palta'."""
    consulta_normalizada = _normalizar_recetas(consulta)
    requeridos, excluidos = set(), set()
    for ingrediente in CATALOGO:
        # Aporte del equipo: "pan de X" no confirma el ingrediente genérico "pan".
        if ingrediente == "pan" and re.search(r"\bpan\s+de\s+\w+", consulta_normalizada):
            continue
        for forma in _formas_ingrediente_receta(ingrediente):
            patron = rf"(?<!\w){re.escape(forma)}(?!\w)"
            for m in re.finditer(patron, consulta_normalizada):
                prefijo = consulta_normalizada[:m.start()]
                negado = re.search(
                    r"(?:\bsin|\bexcepto|\bevitar|\bexcluir|\bno usar)\s+"
                    r"(?:(?:de|el|la|los|las)\s+)?$", prefijo,
                ) or re.search(
                    r"\b(?:sin|excepto|evitar|excluir|no usar)\b[^,;]*\bni\s+(?:(?:de|el|la|los|las)\s+)?$",
                    prefijo,
                )
                (excluidos if negado else requeridos).add(ingrediente)
    return requeridos - excluidos, excluidos


@tool
def buscar_recetas(ingrediente: str) -> str:
    """Busca recetas por ingredientes citados, con acentos y restricciones simples."""
    requeridos, excluidos = _ingredientes_receta_en_consulta(ingrediente)
    if not requeridos:
        return "Indica al menos un ingrediente para buscar recetas, por ejemplo palta o quinoa."
    encontradas = [
        r for r in RECETAS
        if requeridos.issubset(set(r["ingredientes"]))
        and not (excluidos & set(r["ingredientes"]))
    ]
    if not encontradas:
        return "Sin recetas compatibles con todos los ingredientes o exclusiones solicitados."
    return "\n".join(
        f"- {r['nombre']} ({r['minutos']} min) | {r['url']} | usa: {', '.join(r['ingredientes'])}"
        for r in encontradas
    )

@tool
def productos_y_precios() -> str:
    """Catalogo de la tienda con precios en CLP."""
    return "\n".join(f"- {p}: ${v:,}".replace(",", ".") for p, v in CATALOGO.items())

@tool
def estado_pedido(numero: str) -> str:
    """Estado de un pedido por su numero (ej: 1042)."""
    p = PEDIDOS.get(str(numero).strip())
    if not p:
        return f"No existe el pedido {numero}. Pide al cliente verificar el numero."
    return f"Pedido {numero}: {p['estado']} ({p['detalle']}). {p['items']} productos."

@tool
def horarios_despacho(comuna: str) -> str:
    """Horarios de despacho para una comuna."""
    c = comuna.lower()
    for nombre, horario in HORARIOS.items():
        if nombre in c or c in nombre:
            return f"Despacho en {nombre.title()}: {horario}"
    return "Comunas con despacho: " + ", ".join(n.title() for n in HORARIOS) + ". Pregunta por una de ellas."

@tool
def consultar_ofertas(consulta: str) -> str:
    """Consulta solo ofertas ficticias aprobadas para el taller, no promociones reales."""
    import re
    import unicodedata
    from decimal import Decimal, ROUND_HALF_UP

    def normalizar(valor):
        descompuesto = unicodedata.normalize("NFD", str(valor).casefold())
        return "".join(c for c in descompuesto if unicodedata.category(c) != "Mn")

    q = normalizar(consulta)
    variantes = {"palta": ("palta", "paltas"), "tomate": ("tomate", "tomates"),
                 "quinoa": ("quinoa", "quinoas")}
    productos_mencionados = [
        nombre for nombre in CATALOGO
        if any(re.search(rf"(?<!\w){re.escape(forma)}(?!\w)", q)
               for forma in variantes.get(nombre, (normalizar(nombre),)))
    ]
    prefiere_vegetariano = bool(re.search(r"\bvegetarian\w*\b", q))
    consulta_producto_especifico = bool(
        re.search(r"\b(?:ofertas?|promociones?|descuentos?)\s+(?:de|para)\s+", q)
    )
    pide_ofertas_genericas = bool(re.search(r"\b(?:ofertas?|promociones?|descuentos?)\b", q))

    coincidencias = []
    for oferta in OFERTAS_SIMULADAS:
        producto = oferta["producto"]
        tags = [normalizar(tag) for tag in oferta.get("etiquetas_prueba", [])]
        coincide_producto = not productos_mencionados or producto in productos_mencionados
        coincide_tag = not prefiere_vegetariano or any("vegetarian" in tag for tag in tags)
        if productos_mencionados or prefiere_vegetariano:
            coincide = coincide_producto and coincide_tag
        elif consulta_producto_especifico:
            coincide = False
        else:
            coincide = pide_ofertas_genericas
        if coincide:
            coincidencias.append(oferta)

    if not coincidencias:
        return (
            "Sin coincidencias en las ofertas SIMULADAS PARA EL TALLER. "
            "Esto no confirma ni niega promociones comerciales reales ni disponibilidad."
        )

    lineas = [
        "OFERTAS SIMULADAS PARA EL TALLER: datos ficticios; no representan promociones vigentes ni disponibilidad comercial."
    ]
    for oferta in coincidencias:
        producto = oferta["producto"]
        precio_base = CATALOGO.get(producto)
        if precio_base is None:
            continue
        porcentaje = int(oferta["descuento_pct"])
        precio_simulado = int(
            (Decimal(precio_base) * Decimal(100 - porcentaje) / Decimal(100))
            .quantize(Decimal("1"), rounding=ROUND_HALF_UP)
        )
        base_txt = f"${precio_base:,}".replace(",", ".")
        simulado_txt = f"${precio_simulado:,}".replace(",", ".")
        lineas.append(
            f"{producto}: precio base de catalogo {base_txt} CLP; "
            f"descuento simulado {porcentaje}%; precio simulado {simulado_txt} CLP."
        )
        if oferta.get("etiquetas_prueba"):
            lineas.append(
                "Etiqueta ficticia de prueba: vegetariana; no es certificacion alimentaria."
            )
    return "\n".join(lineas)


In [ ]:
# ═══════════════ ESTADO Y ROUTER (ya listos) ═══════════════════════════════
class Estado(TypedDict):
    entrada: str
    ruta: str
    respuesta: str

class Ruta(BaseModel):
    destino: Literal["receta", "pedido", "horarios", "conversar", "ofertas"] = Field(
        description="Flujo que atiende el mensaje del cliente.")

def _es_503(exc):
    """Reconoce exclusivamente HTTP 503 en el error o su cadena de causas."""
    vistos, actual = set(), exc
    while actual is not None and id(actual) not in vistos:
        vistos.add(id(actual))
        if getattr(actual, "status_code", None) == 503 or getattr(actual, "code", None) == 503:
            return True
        actual = getattr(actual, "__cause__", None) or getattr(actual, "__context__", None)
    return False

def invocar_gemini(funcion, *args, **kwargs):
    """Reintenta solo HTTP 503: dos reintentos adicionales, exponencial+jitter."""
    import random, time
    for intento in range(3):
        try:
            return funcion(*args, **kwargs)
        except Exception as exc:
            if not _es_503(exc) or intento == 2:
                raise
            time.sleep((0.2 * (2 ** intento)) + random.uniform(0, 0.15))

def _ruta_respaldo(entrada):
    """Fallback determinista solo si hay una intenci?n comercial inequ?voca."""
    import unicodedata
    t = "".join(c for c in unicodedata.normalize("NFD", str(entrada).lower()) if unicodedata.category(c) != "Mn")
    hits = []
    # Fallback determinista: solo se?ales culinarias expl?citas o varios ingredientes.
    # Una menci?n aislada de producto no basta para asumir que pide una receta.
    verbos_preparacion = (" cocinar", "cocina", "preparar", "prepara", "preparame", "hacer", "hazme", "armar", "mezclar", "hornear", "cortar", "quiero cocinar", "como preparo", "que preparo")
    ingredientes_culinarios = ("tomate", "tomates", "jitomate", "albahaca", "palta", "aguacate", "quinoa", "queso", "pasta", "salmon", "limon", "pan", "aceite de oliva", "eneldo", "tortilla")
    menciones_ingrediente = {x for x in ingredientes_culinarios if re.search(rf"(?<!\w){re.escape(x)}(?!\w)", t)}
    if any(x in t for x in ("receta", "recetas")) or any(x in t for x in verbos_preparacion) or len(menciones_ingrediente) >= 2:
        hits.append("receta")
    if "pedido" in t or "orden" in t: hits.append("pedido")
    if any(x in t for x in ("horario", "despacho", "comuna")): hits.append("horarios")
    if any(x in t for x in ("oferta", "ofertas", "descuento", "promocion")): hits.append("ofertas")
    return hits[0] if len(hits) == 1 else ("conversar" if not hits else None)

def router(estado: Estado):
    """El supervisor: clasifica la intencion y delega (Demo 9)."""
    try:
        d = invocar_gemini(llm.with_structured_output(Ruta).invoke,
            f"{PROMPT_ROUTER}\n\nMensaje del cliente: {estado['entrada']}").destino
    except Exception as exc:
        if not _es_503(exc):
            raise
        d = _ruta_respaldo(estado.get("entrada", ""))
        if d is None:
            print("[router] 503; intencion ambigua, se solicita aclaracion")
            return {"ruta": "conversar", "fallback_503": True, "respuesta": "No puedo determinar con seguridad qué necesitas ahora. ¿Quieres una receta, consultar un pedido, horarios u ofertas?"}
        print(f"[router] fallback_503 determinista: {d}")
        return {"ruta": d, "fallback_503": True}
    print(f"[router] ruta: {d}")
    return {"ruta": d}


In [ ]:
# ═══════════════ NODO MODELO (LISTO) — imita este patron ═══════════════════
def conversar(estado: Estado):
    """FLUJO CONVERSACIONAL: responde con memoria del chat.

    La plataforma inyecta el historial en CHAT_HISTORY: pasarselo al LLM
    es lo que convierte respuestas sueltas en una CONVERSACION."""
    historial = os.environ.get("CHAT_HISTORY", "[]")
    try:
        r = invocar_gemini(llm.invoke,f"{PROMPT_ASISTENTE}\n\nHistorial de la conversacion: {historial}\n"
                   f"Cliente: {estado['entrada']}")
    except Exception as exc:
        if not _es_503(exc):
            raise
        return {"respuesta": "El asistente conversacional no está disponible temporalmente. Puedo ayudarte con recetas, pedidos, horarios u ofertas.", "fallback_503": True}
    return {"respuesta": texto(r)} # Ejemplo del TODO 4


In [ ]:
# ═══════════════ TODO 1 · FLUJO DE RECETAS ═════════════════════════════════
def receta(estado: Estado):
    """Recomienda una receta de la herramienta y precios respaldados por catálogo."""
    solicitud = str(estado.get("entrada", "")).strip()
    opciones = buscar_recetas.invoke({"ingrediente": solicitud})
    precios = productos_y_precios.invoke({})

    # Leer solamente recetas, URLs, ingredientes y precios devueltos por las tools.
    recetas_disponibles = []
    for linea in str(opciones).splitlines():
        m = re.match(r"- (.+?) \((\d+) min\) \| (https?://\S+) \| usa: (.+)$", linea.strip())
        if m:
            recetas_disponibles.append({
                "nombre": m.group(1).strip(),
                "url": m.group(3).strip(),
                "minutos": int(m.group(2)),
                "ingredientes": [i.strip() for i in m.group(4).split(",")],
            })
    catalogo = {}
    for linea in str(precios).splitlines():
        m = re.match(r"- (.+?): \$(\d[\d.]*)$", linea.strip())
        if m:
            catalogo[m.group(1).strip().casefold()] = int(m.group(2).replace(".", ""))
    # Recuperar el fallback hist?rico solo ante b?squeda vac?a: las dos
    # primeras recetas del cat?logo original, nunca como coincidencias del t?rmino.
    usar_fallback = not recetas_disponibles
    if usar_fallback:
        recetas_disponibles = [
            {"nombre": r["nombre"], "url": r["url"], "minutos": r["minutos"],
             "ingredientes": list(r["ingredientes"])}
            for r in RECETAS[:2]
        ]

    solicitud_norm = _normalizar_recetas(solicitud)
    requeridos, excluidos = _ingredientes_receta_en_consulta(solicitud)
    es_vegana = bool(re.search(r"\bvegan[oa]s?\b", solicitud_norm))
    alergia = bool(re.search(r"\balergi\w*\b", solicitud_norm))
    # No hay marcadores de alergenos suficientes para certificar inocuidad.
    if alergia or any(t in solicitud_norm for t in ("celiaco", "celiaca", "sin gluten", "sin lactosa")):
        return {"respuesta": texto("No puedo verificar compatibilidad con esa restricción usando los datos disponibles; no recomendaré una receta sin poder comprobarla.")}

    # Excluir opciones antes de decidir. Ingredientes animales conocidos en el
    # propio cat?logo impiden presentar como vegana una receta incompatible.
    ingredientes_animales = {"queso", "salmon"}
    recetas_disponibles = [
        r for r in recetas_disponibles
        if requeridos.issubset({_normalizar_recetas(x) for x in r["ingredientes"]})
        and not (excluidos & {_normalizar_recetas(x) for x in r["ingredientes"]})
        and not (es_vegana and ingredientes_animales & {_normalizar_recetas(x) for x in r["ingredientes"]})
    ]
    if not recetas_disponibles:
        return {"respuesta": texto("No encontré una receta del catálogo que cumpla todos los ingredientes y restricciones indicados.")}
    if usar_fallback and not requeridos:
        alternativas = "\n".join(
            f"- {r['nombre']}: {r['url']}" for r in recetas_disponibles
        )
        return {"respuesta": texto(
            "No hay una receta con ese ingrediente en el catálogo. "
            "Como alternativas disponibles, puedes revisar:\n" + alternativas
        )}

    # Una opci?n ?nica proviene directamente de la herramienta; evitar una
    # selecci?n redundante del modelo. Con varias, exigir nombre exacto validado.
    if len(recetas_disponibles) == 1:
        receta_elegida = recetas_disponibles[0]
    else:
        instruccion = (
            f"{PROMPT_CHEF}\n\nSolicitud completa del cliente: {solicitud}\n\n"
            f"Recetas disponibles (fuente autorizada):\n{opciones}\n\n"
            f"Productos y precios disponibles (CLP, fuente autorizada):\n{precios}\n\n"
            "Elige una receta de la lista que mejor responda a toda la solicitud. "
            "Responde solo con el nombre exacto de esa receta; no añadas datos nuevos."
        )
        try:
            seleccion_modelo = texto(invocar_gemini(llm.invoke, instruccion)).strip().strip(" \"'`.,;:¡!¿?")
        except Exception as exc:
            if not _es_503(exc):
                raise
            return {"respuesta": "No puedo seleccionar una receta de forma segura porque el servicio está temporalmente no disponible.", "fallback_503": True}
        receta_elegida = next(
            (r for r in recetas_disponibles
             if _normalizar_recetas(r["nombre"]) == _normalizar_recetas(seleccion_modelo)),
            recetas_disponibles[0],  # SIN_COINCIDENCIA o nombre no válido: primera receta compatible de la tool
        )
    if receta_elegida is None:
        return {"respuesta": (
            "No pude validar una receta de las opciones disponibles. "
            "Prueba con otro ingrediente o reformula tu solicitud."
        )}

    # Las cantidades solo se reconocen si el cliente las indicó para cada producto.
    lista = []
    subtotales = []
    cantidades_completas = True
    for producto in receta_elegida["ingredientes"]:
        precio = catalogo.get(producto.casefold())
        if precio is None:
            cantidades_completas = False
            lista.append(f"- {producto}: precio no disponible en el catálogo")
            continue
        formas = sorted({producto, producto + "s", producto + "es"}, key=len, reverse=True)
        cantidad = None
        for forma in formas:
            encontrado = re.search(
                rf"\b(\d+)\s+(?:unidades?\s+)?(?:de\s+)?{re.escape(forma)}\b",
                solicitud, re.IGNORECASE,
            )
            if encontrado:
                cantidad = int(encontrado.group(1))
                break
        precio_txt = f"${precio:,}".replace(",", ".")
        if cantidad is None:
            cantidades_completas = False
            lista.append(f"- {producto}: cantidad no especificada; precio catálogo {precio_txt} CLP")
        else:
            subtotal = cantidad * precio
            subtotales.append(subtotal)
            subtotal_txt = f"${subtotal:,}".replace(",", ".")
            lista.append(f"- {producto}: {cantidad} × {precio_txt} = {subtotal_txt} CLP")
    total = ""
    if cantidades_completas and len(subtotales) == len(receta_elegida["ingredientes"]):
        total_num = sum(subtotales)
        total = f"\nTotal estimado: ${total_num:,} CLP.".replace(",", ".")
    respuesta = (
        f"Te recomiendo {receta_elegida['nombre']}: {receta_elegida['url']}\n"
        f"Tiempo de preparación: {receta_elegida['minutos']} minutos (datos simulados).\n"
        "Lista de compra (precios del catálogo):\n" + "\n".join(lista) + total
    )
    return {"respuesta": texto(respuesta)}

# ═══════════════ TODO 2 · FLUJO ESTADO DE PEDIDOS ══════════════════════════
def pedido(estado: Estado):
    """Consulta el estado exclusivamente a partir del número del pedido."""
    entrada = str(estado.get("entrada", "")).strip()
    # Patrón del equipo: extraer identificadores con contexto explícito, no cantidades.
    def _sin_tildes_pedido(texto_entrada):
        import unicodedata
        d = unicodedata.normalize("NFD", texto_entrada.casefold())
        return "".join(c for c in d if unicodedata.category(c) != "Mn")

    consulta = _sin_tildes_pedido(entrada)
    contexto = re.search(r"\b(?:pedido|orden)\b", consulta)
    numero = None
    if contexto:
        # Aceptar marcadores reconocibles después de pedido/orden, incluso en frases.
        posterior = consulta[contexto.end():][:100]
        marcador = re.search(r"(?:#\s*|\b(?:numero|nro|id)\s*(?:[:#º°]\s*)?)(\d+)\b", posterior)
        if marcador:
            numero = marcador.group(1)
        else:
            inmediato = re.match(r"\s*(?:(?:es\s+)?(?:el\s+)?)(\d+)\b", posterior)
            if inmediato:
                numero = inmediato.group(1)
    if numero is None:
        # Solo un número aislado es un identificador válido sin contexto explícito.
        aislado = re.fullmatch(r"\s*#?\s*(\d+)\s*", consulta)
        if aislado:
            numero = aislado.group(1)
    if numero is None:
        return {"respuesta": "Para consultar el estado, indícame el número del pedido (por ejemplo, #1042)."}

    info = estado_pedido.invoke({"numero": numero})
    if str(info).startswith("No existe"):
        return {"respuesta": (f"No encuentro el pedido {numero}. ¿Puedes revisar el número? "
                              "Lo encuentras en el correo de confirmación de tu compra.")}
    return {"respuesta": texto(info)}

# ═══════════════ TODO 3 · FLUJO DE HORARIOS ════════════════════════════════
def horarios(estado: Estado):
    """Consulta horarios con la tool y solicita aclaración si falta comuna."""
    import unicodedata

    entrada = str(estado.get("entrada", "")).strip()
    def normalizar(valor):
        descompuesto = unicodedata.normalize("NFD", str(valor).casefold())
        return "".join(c for c in descompuesto if unicodedata.category(c) != "Mn")

    consulta = normalizar(entrada)
    # La tool es la fuente de cobertura; no se replica su catálogo en el nodo.
    cobertura = texto(horarios_despacho.invoke({"comuna": "__consulta_cobertura__"}))
    coincidencia = re.search(r"Comunas con despacho: (.+?)\. Pregunta", cobertura)
    comunas = [c.strip() for c in coincidencia.group(1).split(",")] if coincidencia else []
    detectadas = [
        comuna for comuna in comunas
        if re.search(rf"(?<!\w){re.escape(normalizar(comuna))}(?!\w)", consulta)
    ]

    # La consulta completa se envía a la herramienta, normalizada solo para que
    # coincidan Ñuñoa/Maipú con las claves simuladas sin tilde.
    info_consulta = texto(horarios_despacho.invoke({"comuna": consulta}))
    if len(detectadas) > 1:
        respuestas = [texto(horarios_despacho.invoke({"comuna": normalizar(comuna)})) for comuna in detectadas]
        # Identificar nombres no cubiertos en segmentos conectados por y/e/coma.
        segmentos = re.split(r"\s+(?:y|e|,)\s+", consulta)
        desconocidas = []
        for segmento in segmentos:
            candidato = re.sub(r"^(?:horarios?(?: de despacho)?|en|para)\s+", "", segmento).strip(" ??.,")
            if not candidato or any(normalizar(c) in candidato for c in comunas):
                continue
            candidato = re.sub(r"\b(?:horarios?|despacho|tienen|hay)\b", "", candidato).strip(" ??.,")
            if candidato and candidato not in desconocidas:
                desconocidas.append(candidato)
        aviso = ("\nSin cobertura confirmada en los datos simulados para: " + ", ".join(desconocidas) + ".") if desconocidas else ""
        return {"respuesta": texto("Horarios para las comunas indicadas:\n" + "\n".join(respuestas) + aviso)}
    if len(detectadas) == 1:
        segmentos = re.split(r"\s+(?:y|e|,)\s+", consulta)
        desconocidas = []
        for segmento in segmentos:
            candidato = re.sub(r"^(?:horarios?(?: de despacho)?|en|para)\s+", "", segmento).strip(" ??.,")
            candidato = re.sub(r"\b(?:horarios?|despacho|tienen|hay)\b", "", candidato).strip(" ??.,")
            if candidato and not any(normalizar(c) in candidato for c in comunas) and candidato not in desconocidas:
                desconocidas.append(candidato)
        aviso = ("\nSin cobertura confirmada en los datos simulados para: " + ", ".join(desconocidas) + ".") if desconocidas else ""
        return {"respuesta": texto(info_consulta + aviso)}

    if comunas:
        return {"respuesta": texto(
            "No tengo despacho registrado para esa comuna (o no la reconocí). Las comunas con despacho son: "
            + ", ".join(comunas) + "."
        )}
    # Si el formato de cobertura cambia, contestar solo con información de la tool.
    return {"respuesta": texto(info_consulta if info_consulta else cobertura)}

def ofertas(estado: Estado):
    """Devuelve exclusivamente resultados del fixture simulado de ofertas."""
    consulta = str(estado.get("entrada", "")).strip()
    resultado = consultar_ofertas.invoke({"consulta": consulta})
    return {"respuesta": texto(resultado)}


In [ ]:
# ═══════════════ EL GRAFO (ya cableado) ════════════════════════════════════
grafo = StateGraph(Estado)
grafo.add_node("router", router)
grafo.add_node("receta", receta)
grafo.add_node("pedido", pedido)
grafo.add_node("horarios", horarios)
grafo.add_node("conversar", conversar)
grafo.add_node("ofertas", ofertas)

grafo.add_edge(START, "router")
grafo.add_conditional_edges("router", lambda e: e["ruta"],
                            {"receta": "receta", "pedido": "pedido",
                             "horarios": "horarios", "conversar": "conversar",
                             "ofertas": "ofertas"})
for nodo_final in ("receta", "pedido", "horarios", "conversar", "ofertas"):
    grafo.add_edge(nodo_final, END)

app = grafo.compile()

# El mapa estático oficial de LangGraph, como imagen en el chat:
#print(app.get_graph().draw_mermaid_png())

# ═══════════════ TODO 5 (BONUS) · TU FLUJO ADICIONAL ═══════════════════════
# Inventa un flujo nuevo (ofertas de la semana, reclamos, calcular el
# presupuesto de una receta, vinos que maridan...). Necesitas tocar 4 cosas:
#   1. una tool nueva (o datos simulados nuevos)
#   2. un nodo nuevo
#   3. su opcion en el Literal de Ruta (y una pista en PROMPT_ROUTER)
#   4. su entrada en add_conditional_edges + add_edge al END


In [ ]:
# ??????????????? MEMORIA LOCAL DE SESI?N (Jupyter) ??????????????????????????
# conversar() consume CHAT_HISTORY pero no lo acumula. Este wrapper mantiene
# el historial solo en memoria del kernel y utiliza el grafo original.
import json as _json_chat

_chat_session_history = None

def reiniciar_chat_local():
    """Reinicia expl?citamente la conversaci?n local en el kernel actual."""
    global _chat_session_history
    _chat_session_history = []
    os.environ["CHAT_HISTORY"] = "[]"


def _cargar_historial_chat_local():
    """Inicializa la sesi?n una vez, preservando un historial JSON existente."""
    global _chat_session_history
    if _chat_session_history is None:
        try:
            cargado = _json_chat.loads(os.environ.get("CHAT_HISTORY", "[]"))
        except (TypeError, ValueError):
            cargado = []
        _chat_session_history = cargado if isinstance(cargado, list) else []
    return _chat_session_history


def chat_local(entrada: str):
    """Ejecuta un turno por el grafo original y acumula el historial en sesi?n."""
    historial = _cargar_historial_chat_local()
    os.environ["CHAT_HISTORY"] = _json_chat.dumps(historial, ensure_ascii=False)
    estado = {"entrada": str(entrada), "ruta": "", "respuesta": ""}
    resultado = app.invoke(estado)
    historial.extend([
        {"role": "user", "content": estado["entrada"]},
        {"role": "assistant", "content": str(resultado.get("respuesta", ""))},
    ])
    os.environ["CHAT_HISTORY"] = _json_chat.dumps(historial, ensure_ascii=False)
    return resultado


In [ ]:
# ═══════════════ EJECUCION ═════════════════════════════════════════════════
user_input = os.environ.get("USER_INPUT", "hola, que sabes hacer?")
inicial = {"entrada": user_input, "ruta": "", "respuesta": ""}
final = trazar(app, inicial, responder_desde="respuesta")
            